In [ ]:
import sys
import os


RESULTS_PATH = 'results.csv'
OPTUNA_DATABASE = 'study_results.db'
BASE_CONFIGURATION_NAME = 'configuration'


In [ ]:
from automl.loggers.result_logger import ResultLogger
import optuna
import optuna.visualization as vis
from automl.utils.optuna_utils import load_study_from_database
import matplotlib.pyplot as plt


# Load the experiment

In [ ]:
#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-dqn-2\\HPOptimizationExperiments\\3\\experiments"
#experiment_relative_path = 'original'

#base_experiment_path = "C:\\ricardo-goncalo-thesis-project\\project\\examples\\hp_optimization\\data\\hp_lodaded_exps"
#experiment_relative_path = "exp_27"

#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-ppo-multi_thread\\HpExperiments\\1"
#experiment_relative_path = "experiment"

base_experiment_path = "C:\\Experiments\\hyperband_ppo_cartpole\\HpExperiments_v3"
experiment_relative_path = "1"

#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-dqn-2\\HPOptimizationExperiments\\3\\experiments"
#experiment_relative_path = "original_adapted"

experiment_path = f'{base_experiment_path}\\{experiment_relative_path}'



In [ ]:
if not os.path.exists(experiment_path):
    raise Exception("DOES NOT EXIST")

In [ ]:
from automl.core.advanced_input_management import gen_component_from
from automl.hp_opt.hp_optimization_pipeline import HyperparameterOptimizationPipeline


hyperparameter_optimization_pipeline : HyperparameterOptimizationPipeline = gen_component_from(experiment_path)
hyperparameter_optimization_pipeline.change_logger_level("NONE") # we don't want any logging done while we're looking into it


In [ ]:
from automl.hp_opt.hp_opt_strategies.hp_optimization_loader_detached import HyperparameterOptimizationPipelineLoaderDetached

USE_MULTIPLE = isinstance(hyperparameter_optimization_pipeline, HyperparameterOptimizationPipelineLoaderDetached)

# Evaluation of HyperparameterOptimizationPipeline

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_hp_opt_results_logger

hyperparameter_optimization_results = hyperparameter_optimization_pipeline.get_decoupled_results_logger()

print(f"Hyperparameter_optimization_results in path: {hyperparameter_optimization_results.get_artifact_directory()}")

In [ ]:
if not USE_MULTIPLE:
    df = hyperparameter_optimization_results.get_dataframe()
    df["component_index"] = 0

In [ ]:
experiments_in_results = set(hyperparameter_optimization_results.get_dataframe()["experiment"].values)
component_indexes_in_results = set(hyperparameter_optimization_results.get_dataframe()["component_index"].values)

print(f"Experiments in results:\n{experiments_in_results}")

print(f"Component indexes per experiment:\n{component_indexes_in_results}")


In [ ]:
colors_available_for_component_indexes = ["red", "blue", "green"]
colors_for_component_indexes = {}

color_iter = iter(colors_available_for_component_indexes)
for component_index in component_indexes_in_results:
    colors_for_component_indexes[component_index] = next(color_iter)

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_hp_opt_optuna_study


optuna_study = hyperparameter_optimization_pipeline.get_study()


## Hyperparameter Study

In [ ]:
try:
    print(f"optuna_study done with with best value {optuna_study.best_value} in trial {optuna_study.best_trial.number} with best parameters:\n{optuna_study.best_params}")

except:
    print("No best trial yet")

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import print_optuna_trials_info

print_optuna_trials_info(optuna_study)

In [ ]:
fig = vis.plot_intermediate_values(optuna_study)
fig.show()

In [ ]:
fig = vis.plot_optimization_history(optuna_study)
fig.show()

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_evaluation_statistics_per_step


evaluations_stats_per_step = get_evaluation_statistics_per_step(hyperparameter_optimization_results)

for step in evaluations_stats_per_step.keys():

    evauation_at_step = evaluations_stats_per_step[step]

    print(f"{step}: avg_mean: {evauation_at_step['avg_mean']}, avg_srd: {evauation_at_step['avg_std']}")

# Global evaluation of configurations

In [ ]:
AGGREGATE_NUMBER = 10 #the number of neighbor points to sum to plot the needed graphs

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import study_of_configuration      
from automl.hp_opt.hp_eval_results.hp_eval_results import study_of_components_for_configuration      


In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_results_of_configurations_components                          



results_of_configurations : dict[str, ResultLogger] = get_results_of_configurations_components(experiment_path, use_multiple=USE_MULTIPLE)

In [ ]:
print(f"Configurations:  {results_of_configurations.keys()}")

# Global view of performance

In [ ]:
steps_done = hyperparameter_optimization_results.list_of_unique_values("step")
print(steps_done)

In [ ]:

for step in steps_done:

    ax = hyperparameter_optimization_results.plot_overlaid_bar(x_axis="experiment", y_axis='result', fixed_value_tuple=("step", step), colors_for_value=("component_index", colors_for_component_indexes)) 

    #hyperparameter_optimization_results.plot_linear_regression(x_axis=['experiment'], y_axis='result', to_show=False, fixed_value_tuple=("step", step), group_by=['experiment', 'component_index'], ax=ax)
    #hyperparameter_optimization_results.plot_current_graph(title=f'experiments_at_step_{step}')